In [ ]:
import joblib
import numpy as np
from PIL import Image
from pathlib import Path
import cv2
import time
import psutil
import threading
import openflexure_microscope_client as ofm_client
import tracemalloc
import os
import csv

# ============================
#  CPU MONITOR SETUP
# ============================
cpu_usage_list = []
monitoring = True

def monitor_cpu():
    while monitoring:
        cpu = psutil.cpu_percent(interval=0.2)
        cpu_usage_list.append(cpu)

cpu_thread = threading.Thread(target=monitor_cpu, daemon=True)
cpu_thread.start()

# ============================
#  Setup & Model Loading
# ============================
microscope = ofm_client.MicroscopeClient("10.121.29.160")

MODEL_PATH  = "svr_autofocus_tunedv55.pkl"
SCALER_PATH = "scaler_svr_tunedv55.pkl"
regressor = joblib.load(MODEL_PATH)
scaler    = joblib.load(SCALER_PATH)

output_dir = Path.home() / "Desktop" / "microscope_autofocus_two_shot"
output_dir.mkdir(exist_ok=True)

# Fixed configuration from data collection parameters
EXPLORATION_STEP = 200

# ============================
#  Helper: Laplacian variance
# ============================
def compute_variance(img_array):
    gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

# ============================
#  START RAM TRACKING
# ============================
process_mem = psutil.Process(os.getpid())
os_ram_start = process_mem.memory_info().rss
tracemalloc.start()

# ============================
#  1. Capture Image 1 (Baseline / Cold Frame)
# ============================
pos = microscope.position
x_curr = pos['x']
y_curr = pos['y']
z_start = pos['z']

print(f"📍 Starting position: X={x_curr}, Y={y_curr}, Z={z_start}")

t0 = time.time()
img1 = microscope.grab_image()
arr1 = np.array(img1)
Image.fromarray(arr1).save(output_dir / "initial_image_shot1.png")
t_capture1 = time.time() - t0

var1 = compute_variance(arr1)
print(f"🔍 Image 1 Variance: {var1:.4f} (captured in {t_capture1:.2f} s)")

# ============================
#  2. Take the Exploration Step
# ============================
print(f"🔄 Executing exploration step of {EXPLORATION_STEP} units...")
start_explore_move = time.time()

z_explore = z_start + EXPLORATION_STEP
pos['z'] = z_explore
microscope.move(pos)

t_explore_move = time.time() - start_explore_move

# ============================
#  3. Capture Image 2 (Exploration Frame)
# ============================
t1 = time.time()
img2 = microscope.grab_image()
arr2 = np.array(img2)
Image.fromarray(arr2).save(output_dir / "exploration_image_shot2.png")
t_capture2 = time.time() - t1

var2 = compute_variance(arr2)
print(f"🔍 Image 2 Variance: {var2:.4f} (captured in {t_capture2:.2f} s)")

# ============================
#  4. Build Dynamic Feature Vector 
# ============================
variance_ratio = var2 / (var1 + 1e-9)
variance_sq    = var2 ** 2
is_improving   = 1 if var2 > var1 else 0
direction      = 1 if EXPLORATION_STEP >= 0 else -1
dVariance_abs  = abs(var2 - var1)

# Sequence matches training array structure exactly
X_sample = np.array([[z_explore,
                      variance_ratio,
                      variance_sq,
                      is_improving,
                      direction,
                      dVariance_abs]])

X_scaled = scaler.transform(X_sample)

# ============================
#  5. Predict Correction Step
# ============================
start_pred = time.time()

predicted_step = int(round(regressor.predict(X_scaled)[0]))

prediction_time = time.time() - start_pred

print(f"🌲 RF predicted step from exploration point: {predicted_step}")
print(f"⏱ Prediction time: {prediction_time:.4f} s")

# ============================
#  6. Move Microscope to Final Focus
# ============================
start_move = time.time()

z_final = z_explore + predicted_step
pos['z'] = z_final

# Mechanical backlash check
if z_explore > z_final:
    below = max(z_final - 50, 0)
    microscope.move({'x': x_curr, 'y': y_curr, 'z': below})
    time.sleep(0.05)

microscope.move(pos)
move_time = time.time() - start_move

# Calculate combined prediction and move time
pred_move_time = prediction_time + move_time
print(f"🔄 Final Correction Move time: {move_time:.2f} s")
print(f"⚡ Total Prediction + Move time: {pred_move_time:.4f} s")

# ============================
#  7. Capture Final Validation Image
# ============================
t2 = time.time()

final_img = microscope.grab_image()
final_arr = np.array(final_img)
Image.fromarray(final_arr).save(output_dir / "final_focused_image6.png")

t_final = time.time() - t2
var_final = compute_variance(final_arr)

print(f"🔍 Verified Final variance: {var_final:.4f} (captured in {t_final:.2f} s)")
print(f"📈 Net Variance Change: {var_final - var1:+.4f}")

# ============================
#  STOP RAM TRACKING
# ============================
current_py_mem, peak_py_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

os_ram_end = process_mem.memory_info().rss
os_ram_diff = os_ram_end - os_ram_start

# ============================
#  8. Stop CPU Monitoring
# ============================
monitoring = False
cpu_thread.join()

if len(cpu_usage_list) == 0:
    cpu_usage_list.append(psutil.cpu_percent(interval=0.1))

mean_cpu_usage = np.mean(cpu_usage_list)
std_cpu_usage  = np.std(cpu_usage_list)
max_cpu_usage  = np.max(cpu_usage_list)

# ============================
#  9. Summary Printouts
# ============================
print("\n📏 Two-Shot Autofocus Summary")
print(f"   • Base Z position:       {z_start}")
print(f"   • Exploration Z position:  {z_explore}")
print(f"   • Final Predicted Focus Z: {z_final}")
print(f"   • Total travel correction: {z_final - z_start} steps (= {abs(z_final - z_start)*50} nm)")

total_time = t_capture1 + t_explore_move + t_capture2 + pred_move_time + t_final

print("\n⏱ Detailed Processing Profile")
print(f"   • Initial frame capture:   {t_capture1:.2f} s")
print(f"   • Exploration transit:     {t_explore_move:.2f} s")
print(f"   • Exploration capture:     {t_capture2:.2f} s")
print(f"   • Prediction + Move time:  {pred_move_time:.4f} s")
print(f"   • Validation frame capture: {t_final:.2f} s")
print(f"   • ✅ Total Cycle Runtime:  {total_time:.2f} s")

print("\n💻 Core CPU System Diagnostics")
print(f"   • Mean Engine Overhead: {mean_cpu_usage:.2f}%")
print(f"   • Load Dev Standard:    {std_cpu_usage:.2f}%")
print(f"   • Peak Spike Max:       {max_cpu_usage:.2f}%")

peak_py_mem_mb = peak_py_mem / (1024 * 1024)
os_ram_diff_mb = os_ram_diff / (1024 * 1024)

print("\n🧠 Memory Usage Summary")
print(f"   • Peak Python memory:   {peak_py_mem_mb:.2f} MB")
print(f"   • Net OS RAM footprint: {os_ram_diff_mb:.2f} MB")

# ============================
#  10. Save Results to CSV
# ============================
csv_file = output_dir / "results.csv"
file_exists = csv_file.exists()

variance_change = var_final - var1

with open(csv_file, mode='a', newline='') as file:
    writer = csv.writer(file)
    
    # Write the header only if the file is being created for the first time
    if not file_exists:
        writer.writerow([
            "Z_Initial", "Z_Explore", "Z_Final", "Predicted_Step",
            "Initial_Variance", "Explore_Variance", "Final_Variance", "Variance_Change",
            "Prediction_Move_Time_s", "Total_Execution_Time_s", 
            "Mean_CPU_Usage_percent", "Max_CPU_Usage_percent",
            "Peak_Py_Mem_MB", "Net_OS_RAM_MB"
        ])
    
    # Write the data for this specific run
    writer.writerow([
        z_start, z_explore, z_final, predicted_step,
        f"{var1:.4f}", f"{var2:.4f}", f"{var_final:.4f}", f"{variance_change:.4f}",
        f"{pred_move_time:.4f}", f"{total_time:.2f}", 
        f"{mean_cpu_usage:.2f}", f"{max_cpu_usage:.2f}",
        f"{peak_py_mem_mb:.2f}", f"{os_ram_diff_mb:.2f}"
    ])

print(f"\n📁 Results saved successfully to {csv_file}")